# Collect DiD-BCF paper metrics

Upload the small `didbcf_paper_*.zip` result files when available. This notebook automatically includes the 264 verified earlier fits, validates identities and checksums, reports missing jobs, and exports all tables in one ZIP. Partial collections are clearly marked. This collection covers the main rerun and eight budget checks; the remaining pretrend, application and sensitivity campaign follows those checks.


In [ ]:
import os, sys, subprocess, time
from pathlib import Path
assert (3, 12) <= sys.version_info[:2] <= (3, 14), "Use Python 3.12 to 3.14"
SESSION_STARTED = time.time()
DEADLINE = SESSION_STARTED + 9.5 * 3600
for key in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS", "NUMEXPR_NUM_THREADS"):
    os.environ[key] = "1"
SOURCE_COMMIT = '3078e2ca1155b932feab92c722d697803370b8c7'
REPO = Path("/content/didbcf_paper_repo") if Path("/content").exists() else Path.cwd() / "didbcf_paper_repo"
if not (REPO / ".git").exists():
    subprocess.run(["git", "clone", "--filter=blob:none", "--no-checkout", "https://github.com/hugogobato/DiD-BCF.git", str(REPO)], check=True)
subprocess.run(["git", "-C", str(REPO), "fetch", "origin", SOURCE_COMMIT], check=True)
subprocess.run(["git", "-C", str(REPO), "sparse-checkout", "set", 'Simulation_Studies_Revision/Proper_Prior_Rerun/Paper_Rerun_20261011'], check=True)
subprocess.run(["git", "-C", str(REPO), "checkout", "--detach", SOURCE_COMMIT], check=True)
actual = subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip()
assert actual == SOURCE_COMMIT
PACKAGE = REPO / 'Simulation_Studies_Revision/Proper_Prior_Rerun/Paper_Rerun_20261011'
import json
manifest = json.loads((PACKAGE / "manifest.json").read_text())
requirements = [name + "==" + version for name, version in manifest["dependencies"].items()]
subprocess.run([sys.executable, "-m", "pip", "install", "--disable-pip-version-check", *requirements], check=True)
HELPERS = PACKAGE / "helpers"
print("Pinned sources and dependencies installed:", SOURCE_COMMIT)


In [ ]:
from google.colab import files
uploaded = files.upload()
INPUTS = [Path(name).resolve() for name in uploaded if name.endswith(".zip")]
RESULTS = Path("/content/didbcf_paper_tables")
subprocess.run([sys.executable, str(HELPERS / "collect.py"), *map(str, INPUTS), "--out", str(RESULTS)], check=True)
status = json.loads((RESULTS / "status.json").read_text())
print("Main fits:", status["main_completed"], "/", status["main_expected"])
print("Bridge checks:", status["bridge_completed"], "/", status["bridge_expected"])
import shutil
output_file = shutil.make_archive("/content/didbcf_paper_tables", "zip", RESULTS)
try:
    from google.colab import files
    files.download(output_file)
    print("Downloaded:", output_file)
except Exception as e:
    print("(Not on Colab / download skipped):", e)
